In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalMaxPooling1D, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [5]:
df = pd.read_csv("medical_dataset.csv")

print(df.head())

X = df["symptoms"]
y = df["condition"]

           condition                                           symptoms  \
0       Hypertension  The patient experiences frequent headaches, di...   
1            Malaria  High fever with shaking chills followed by swe...   
2    Gastroenteritis  Frequent loose stools with stomach pain, dehyd...   
3  Allergic Rhinitis  Patient experiences sneezing, runny nose, nasa...   
4          Sinusitis  Patient has facial pain, nasal congestion, hea...   

                                              causes  \
0  often linked to immune response, infection, or...   
1  often linked to immune response, infection, or...   
2  often linked to immune response, infection, or...   
3  may be caused by infection, genetic factors, o...   
4  often linked to immune response, infection, or...   

                                            warnings  \
0     consult a doctor if symptoms persist or worsen   
1     consult a doctor if symptoms persist or worsen   
2  immediate medical care is required if bre

In [6]:
max_words = 5000
max_len = 150

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

sequences = tokenizer.texts_to_sequences(X)
X_padded = pad_sequences(sequences, maxlen=max_len)

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Number of diseases:", len(encoder.classes_))

Number of diseases: 18


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_padded, y_encoded,
    test_size=0.2,
    random_state=42
)

In [8]:
num_classes = len(encoder.classes_)

model = Sequential([
    Embedding(max_words,128,input_length=max_len),

    SpatialDropout1D(0.3),

    Bidirectional(LSTM(64,return_sequences=True)),

    GlobalMaxPooling1D(),

    Dense(128,activation="relu"),
    Dropout(0.4),

    Dense(64,activation="relu"),
    Dropout(0.3),

    Dense(num_classes,activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

c:\Users\saram\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test,y_test)
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 61ms/step - accuracy: 0.5320 - loss: 1.4963 - val_accuracy: 0.9987 - val_loss: 0.0227
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 61ms/step - accuracy: 0.9764 - loss: 0.0968 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 58ms/step - accuracy: 0.9872 - loss: 0.0416 - val_accuracy: 1.0000 - val_loss: 8.5009e-04
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 54ms/step - accuracy: 0.9931 - loss: 0.0270 - val_accuracy: 0.9987 - val_loss: 0.0019
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - accuracy: 0.9947 - loss: 0.0225 - val_accuracy: 0.9981 - val_loss: 0.0074
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 14s 69ms/step - accuracy: 0.9962 - loss: 0.0129 - val_accuracy: 1.0000 - val_loss: 4.2154e-04
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 79ms/step - accuracy: 0.9958 - loss: 0.0158 - val_accuracy: 0.9994 - val_loss: 0.0020
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - accuracy: 0.9972 - loss: 0

In [13]:
def predict_disease(text):

    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq,maxlen=max_len)

    pred = model.predict(padded)

    top3 = np.argsort(pred[0])[-3:][::-1]

    for i in top3:
        print(
            encoder.inverse_transform([i])[0],
            "prob:",
            float(pred[0][i])
        )